# Hinglish ToolBench: LoRA fine-tuning

Runs on a Kaggle GPU (T4 x2 or P100; enable it under *Settings → Accelerator*).

1. Add the `hinglish-toolbench-bundle` dataset (built with `scripts/make_kaggle_bundle.sh`) via *Add Input*.
2. Run all cells. Zero-shot baseline, fine-tuning and evaluation predictions are written to `/kaggle/working`.
3. Download `results.zip` and score locally with `htb eval`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q -U peft accelerate

In [ ]:
import glob
import os

BUNDLE = os.path.dirname(glob.glob('/kaggle/input/**/finetune.py', recursive=True)[0])
DATA = os.path.dirname(glob.glob('/kaggle/input/**/train.jsonl', recursive=True)[0])
MODEL = 'Qwen/Qwen3-1.7B'          # also try 'Qwen/Qwen3-0.6B' for the size ablation
RUN = MODEL.split('/')[-1].lower() + '-lora'
OUT = f'/kaggle/working/{RUN}'
print(BUNDLE, DATA, MODEL)
for f in ['train', 'dev', 'test']:
    p = f'{DATA}/{f}.jsonl'
    if os.path.exists(p):
        with open(p) as fh:
            print(f, sum(1 for _ in fh))
    else:
        print(f, 'missing')

## 1. Zero-shot baseline (same prompt, no fine-tuning)

In [ ]:
!python {BUNDLE}/predict.py --model {MODEL} --data {DATA}/test.jsonl \
    --out /kaggle/working/preds/{MODEL.split('/')[-1].lower()}-zeroshot.jsonl --batch-size 8

## 2. LoRA fine-tuning

In [ ]:
!python {BUNDLE}/finetune.py --model {MODEL} --train {DATA}/train.jsonl --dev {DATA}/dev.jsonl \
    --out {OUT} --epochs 2 --lr 2e-4 --lora-r 16 --batch-size 4 --grad-accum 4

## 3. Predictions with the fine-tuned adapter

In [ ]:
!python {BUNDLE}/predict.py --model {MODEL} --adapter {OUT}/adapter --data {DATA}/test.jsonl \
    --out /kaggle/working/preds/{RUN}.jsonl --batch-size 8

In [ ]:
!cd /kaggle/working && zip -qr results.zip preds {RUN}/adapter {RUN}/train_summary.json && ls -lh results.zip